In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel(
    r"C:\Users\shubh\Desktop\Project_Analytics\data\raw\acndata_sessions.json.xlsx"
)

print(df.shape)
print(df.columns)

(16304, 27)
Index(['_meta', 'end', 'min_kWh', 'site', 'start', '_items', '_id',
       'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime',
       'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID',
       'timezone', 'userID', 'userInputs', 'WhPerMile', 'kWhRequested',
       'milesRequested', 'minutesAvailable', 'modifiedAt', 'paymentRequired',
       'requestedDeparture', 'userID.1'],
      dtype='object')


In [3]:
acn = df[
    [
        "connectionTime",
        "disconnectTime",
        "doneChargingTime",
        "kWhDelivered",
        "sessionID",
        "stationID",
        "spaceID",
        "siteID"
    ]
].copy()

In [4]:
acn.rename(
    columns={
        "connectionTime": "start_time",
        "disconnectTime": "end_time",
        "doneChargingTime": "charging_end",
        "kWhDelivered": "energy_delivered",
        "stationID": "station_id"
    },
    inplace=True
)

In [5]:
acn["start_time"] = pd.to_datetime(acn["start_time"])
acn["end_time"] = pd.to_datetime(acn["end_time"])
acn["charging_end"] = pd.to_datetime(acn["charging_end"])

In [6]:
acn["session_duration_hr"] = (
    acn["end_time"] - acn["start_time"]
).dt.total_seconds() / 3600

In [7]:
acn["charging_time_hr"] = (
    acn["charging_end"] - acn["start_time"]
).dt.total_seconds() / 3600

In [8]:
acn["revenue"] = acn["energy_delivered"] * 15

In [9]:
acn["hour"] = acn["start_time"].dt.hour

acn["day_of_week"] = acn["start_time"].dt.day_name()

acn["month"] = acn["start_time"].dt.month

acn["is_weekend"] = (
    acn["start_time"].dt.weekday >= 5
).astype(int)

In [10]:
acn_clean = acn.dropna(
    subset=[
        "start_time",
        "end_time",
        "energy_delivered",
        "station_id"
    ]
).copy()

acn_clean = acn_clean.dropna(
    subset=[
        "charging_end",
        "charging_time_hr"
    ]
)

In [11]:
acn_clean["utilization_rate"] = (
    acn_clean["charging_time_hr"] / 24
)

acn_clean["utilization_rate"] = (
    acn_clean["utilization_rate"]
    .clip(upper=1)
)

In [12]:
queue_proxy = (
    acn_clean
    .groupby(
        ["station_id", "hour"]
    )
    .size()
    .reset_index(name="queue_proxy")
)

acn_clean = acn_clean.merge(
    queue_proxy,
    on=["station_id", "hour"],
    how="left"
)

In [13]:
acn_clean["location_type"] = "workplace"

In [14]:
acn_final = acn_clean[
[
    "start_time",
    "station_id",
    "location_type",
    "energy_delivered",
    "session_duration_hr",
    "charging_time_hr",
    "utilization_rate",
    "revenue",
    "queue_proxy",
    "hour",
    "day_of_week",
    "month",
    "is_weekend"
]
]

In [15]:
acn_final.to_csv(
    "../processed/acn_clean.csv",
    index=False
)

Above ACN Dataset was cleaned and below UrbanEV Dataset 

In [16]:
volume = pd.read_csv("../data/raw/volume.csv")

occupancy = pd.read_csv("../data/raw/occupancy.csv")

duration = pd.read_csv("../data/raw/duration.csv")

price = pd.read_csv("../data/raw/price.csv")

time_df = pd.read_csv("../data/raw/time.csv")

In [17]:
time_df["timestamp_real"] = pd.to_datetime(
    time_df[
        [
            "year",
            "month",
            "day",
            "hour",
            "minute",
            "second"
        ]
    ]
)

In [19]:
volume_long = volume.melt(
    id_vars="timestamp",
    var_name="station_id",
    value_name="energy_delivered"
)

occupancy_long = occupancy.melt(
    id_vars="timestamp",
    var_name="station_id",
    value_name="occupancy"
)

duration_long = duration.melt(
    id_vars="timestamp",
    var_name="station_id",
    value_name="session_duration_hr"
)

price_long = price.melt(
    id_vars="timestamp",
    var_name="station_id",
    value_name="price"
)

In [20]:
urban = volume_long.merge(
    occupancy_long,
    on=["timestamp", "station_id"]
)

urban = urban.merge(
    duration_long,
    on=["timestamp", "station_id"]
)

urban = urban.merge(
    price_long,
    on=["timestamp", "station_id"]
)

In [21]:
time_map = time_df.reset_index()

time_map["timestamp"] = (
    time_map.index + 1
)

urban = urban.merge(
    time_map[
        ["timestamp", "timestamp_real"]
    ],
    on="timestamp"
)

In [22]:
urban["location_type"] = "urban_public"

urban["revenue"] = (
    urban["energy_delivered"]
    *
    urban["price"]
)

urban["utilization_rate"] = (
    urban["occupancy"]
    /
    urban["occupancy"].max()
)

urban["queue_proxy"] = (
    urban["occupancy"]
)

In [23]:
urban["hour"] = (
    urban["timestamp_real"].dt.hour
)

urban["day_of_week"] = (
    urban["timestamp_real"]
    .dt.day_name()
)

urban["month"] = (
    urban["timestamp_real"].dt.month
)

urban["is_weekend"] = (
    urban["timestamp_real"]
    .dt.weekday >= 5
).astype(int)

In [24]:
urban_final = urban[
[
    "timestamp_real",
    "station_id",
    "location_type",
    "energy_delivered",
    "session_duration_hr",
    "utilization_rate",
    "revenue",
    "queue_proxy",
    "hour",
    "day_of_week",
    "month",
    "is_weekend"
]
].copy()

urban_final.rename(
    columns={
        "timestamp_real": "start_time"
    },
    inplace=True
)

In [25]:
urban_final.to_csv(
    "../processed/urbanev_clean.csv",
    index=False
)

In [26]:
master = pd.concat(
    [acn_final, urban_final],
    ignore_index=True
)

In [27]:
print(master.shape)

master.head()

(2149071, 13)


,start_time,station_id,location_type,energy_delivered,session_duration_hr,charging_time_hr,utilization_rate,revenue,queue_proxy,hour,day_of_week,month,is_weekend
0,2018-04-25 11:08:04,2-39-78-362,workplace,7.932,2.201667,2.218333,0.092431,118.980,2,11.0,Wednesday,4.0,0
1,2018-04-25 13:45:10,2-39-95-27,workplace,10.013,11.185000,2.984722,0.124363,150.195,46,13.0,Wednesday,4.0,0
2,2018-04-25 13:45:50,2-39-79-380,workplace,5.257,9.315278,1.098333,0.045764,78.855,81,13.0,Wednesday,4.0,0
3,2018-04-25 14:37:06,2-39-79-379,workplace,5.177,9.307778,1.471111,0.061296,77.655,38,14.0,Wednesday,4.0,0
4,2018-04-25 14:40:34,2-39-79-381,workplace,10.119,8.377222,2.998889,0.124954,151.785,52,14.0,Wednesday,4.0,0


In [28]:
master.to_csv(
    "../processed/master_features.csv",
    index=False
)